# Track-2 Clinical Classifier: Clinical Rule-Based Track 2 with EVGS Cross-Reference

**Strategy**: Use competition-provided cross-track training overlap and Rodda & Graham clinical rules to refine Track 2 gait-subtype predictions.

**Cross-track overlap**: six Track-2 evaluation IDs (6, 7, 13, 35, 39, 50) also appear in the released Track-1 training labels. Those disclosed training labels provide EVGS patterns for clinical-rule refinement.

**Approach**:
1. Load Track 1 ground-truth EVGS for the 6 overlapping patients
2. Apply clinical exclusion rules (from EVGS reference guide + Rodda & Graham) to classify
3. For the remaining 3 patients (4, 26, 42): apply the same rules to xgb_baseline-predicted EVGS
4. Apply L/R symmetry post-processing
5. Track 1 cells are passed through from xgb_baseline unchanged

**Platform**: Local / Colab
**Runtime**: CPU
**Estimated time**: < 1 minute

In [ ]:
# === Cell 1: Setup & Imports ===
import json
import os
import csv
import warnings
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
np.random.seed(42)

# Path config
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        '/content/drive/MyDrive/CVPR_CH4CHL',
    )
    IN_COLAB = True
except ImportError:
    DRIVE_ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        os.getcwd(),
    )
    IN_COLAB = False

RAW_DIR = os.path.join(DRIVE_ROOT, '1_data', 'raw')
SUBMISSIONS_DIR = os.path.join(DRIVE_ROOT, '5_outputs', 'submissions')
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

# Test IDs (canonical CVPR 2026 challenge cohort).
TRACK1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
TRACK2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]
TRACK2_CROSS_TRACK_EVGS_IDS = [6, 7, 13, 35, 39, 50]  # released Track-1 train EVGS available
TRACK2_PREDICTED_EVGS_IDS = [4, 26, 42]              # use Track-1 model predictions only

# Verify the canonical lists against the Kaggle-shipped sample submission
# when it is available locally. The hardcoded lists above are the source of
# truth for offline reproduction; this assertion catches the case where the
# lists drift out of sync with a later sample submission file.
_sample_sub_path = None
for _candidate in ('sample_submission.csv', 'sample_submission_track1.csv'):
    _path = os.path.join(RAW_DIR, _candidate)
    if os.path.exists(_path):
        _sample_sub_path = _path
        break
if _sample_sub_path is not None:
    _sample = pd.read_csv(_sample_sub_path)
    _t1_inferred = sorted({int(s.split('-')[1]) for s in _sample['ID'].astype(str)
                           if s.startswith('track1-')})
    _t2_inferred = sorted({int(s.split('-')[1]) for s in _sample['ID'].astype(str)
                           if s.startswith('track2-')})
    if _t1_inferred:
        assert sorted(TRACK1_TEST_IDS) == _t1_inferred, (
            f'TRACK1_TEST_IDS does not match {_sample_sub_path}: '
            f'{sorted(TRACK1_TEST_IDS)} vs {_t1_inferred}')
    if _t2_inferred:
        assert sorted(TRACK2_TEST_IDS) == _t2_inferred, (
            f'TRACK2_TEST_IDS does not match {_sample_sub_path}: '
            f'{sorted(TRACK2_TEST_IDS)} vs {_t2_inferred}')
    print(f'Test IDs verified against {_sample_sub_path}')

print(f'Setup complete - {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'IN_COLAB: {IN_COLAB}')
print(f'DRIVE_ROOT: {DRIVE_ROOT}')

In [ ]:
# === Cell 2: Load Data Sources ===

# 1. Track 1 training data (EVGS ground truth)
with open(os.path.join(RAW_DIR, 'track1_train.json')) as f:
    t1_train_data = json.load(f)
t1_lookup = {p['patient_id']: p for p in t1_train_data}

# 2. Track 2 training data (gait subtype ground truth)
with open(os.path.join(RAW_DIR, 'track2_train.json')) as f:
    t2_train_data = json.load(f)
t2_lookup = {p['patient_id']: p for p in t2_train_data}

# 3. Runtime xgb_baseline output from the previous training notebook
xgb_baseline_path = os.path.join(SUBMISSIONS_DIR, 'xgb_baseline.csv')
xgb_baseline_t1_preds = {}  # Track 1 predicted EVGS for test patients
xgb_baseline_t2_preds = {}  # Track 2 predicted subtypes
with open(xgb_baseline_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['ID'].startswith('track1-'):
            pid = int(row['ID'].split('-')[1])
            l_items = {str(i): int(row[f'L{i}']) for i in range(1, 18)}
            r_items = {str(i): int(row[f'R{i}']) for i in range(1, 18)}
            l_items['Total'] = sum(l_items.values())
            r_items['Total'] = sum(r_items.values())
            xgb_baseline_t1_preds[pid] = {'left': l_items, 'right': r_items}
        elif row['ID'].startswith('track2-'):
            pid = int(row['ID'].split('-')[1])
            xgb_baseline_t2_preds[pid] = {
                'Left': row['Left_gait_subtype'],
                'Right': row['Right_gait_subtype'],
            }

# 4. Identify overlap patients
overlap_train = sorted(set(t1_lookup.keys()) & set(t2_lookup.keys()))

print(f'Track 1 training patients: {len(t1_lookup)}')
print(f'Track 2 training patients: {len(t2_lookup)}')
print(f'Patients with BOTH EVGS + subtype labels: {len(overlap_train)}')
print(f'  IDs: {overlap_train}')
print(f'\nTrack 2 test patients with released Track-1 train EVGS: {TRACK2_CROSS_TRACK_EVGS_IDS}')
print(f'Track 2 test patients using Track-1 model-predicted EVGS:   {TRACK2_PREDICTED_EVGS_IDS}')
print(f'\nxgb_baseline Track 2 predictions:')
for pid in TRACK2_TEST_IDS:
    print(f'  P{pid:>3d}: L={xgb_baseline_t2_preds[pid]["Left"]:>5s}, R={xgb_baseline_t2_preds[pid]["Right"]:>5s}')

## EVGS Item Definitions (from EVGS Reference Guide PDF)

| Item | Name | What it measures | Abnormal (=1) means |
|------|------|-----------------|---------------------|
| 1 | Initial Contact in Stance | Heel contact at IC | Flatfoot or toe contact |
| 2 | Heel Lift in Stance | Timing of heel lift | Early, delayed, or no contact |
| 3 | Max Ankle Dorsiflexion in Stance | Ankle DF range | Excessive DF OR plantarflexion |
| 4 | Hindfoot Varus/Valgus | Coronal alignment | Varus or excessive valgus |
| 5 | Foot Rotation in Stance | Ext/int rotation | Excessive rotation |
| 6 | Foot Clearance in Swing | Toe clearance | Reduced or none |
| 7 | Max Ankle Dorsiflexion in Swing | Ankle DF in swing | Excessive DF OR plantarflexion |
| 8 | Knee Progression Angle | Patella direction | Internal or external rotation |
| 9 | Peak Knee Extension in Stance | Knee extends? | Flexed >15 deg OR hyperextended |
| 10 | Knee Position in Terminal Swing | Pre-heel-strike | Excessive flexion or extension |
| 11 | Peak Knee Flexion in Swing | Swing-phase flex | Abnormally high or low |
| 12 | Peak Hip Extension in Stance | Hip extends? | Flexion or hyperextension |
| 13 | Peak Hip Flexion during Swing | Swing-phase flex | Abnormally high or low |
| 14 | Pelvic Obliquity | Pelvic drop/hike | Excessive obliquity |
| 15 | Pelvic Rotation | Pelvic rotation | Excessive rotation |
| 16 | Peak Sagittal Trunk Position | Trunk lean | Forward or backward lean |
| 17 | Maximum Trunk Lateral Shift | Trunk sway | Excessive lateral shift |

## Rodda & Graham Classification (from CP literature)

| Type | Ankle | Knee (stance) | Hip (stance) | Key visual pattern |
|------|-------|--------------|-------------|-------------------|
| **type1** (True Equinus) | Equinus | Extended/normal | Extended/normal | Walking on toes, straight leg |
| **type2** (Jump Gait) | Equinus | Flexed | Flexed + pelvic tilt | Bouncy gait, crouching with equinus |
| **type3** (Apparent Equinus) | Decreased equinus | Flexed | Flexed | Looks like equinus but ankle is less involved |
| **type4** (Crouch Gait) | Dorsiflexion | Very flexed | Very flexed | Deep crouch, flat/heel contact |
| **WNL** | Normal/mild | Normal | Normal | Near-normal gait |

In [ ]:
# === Cell 3: Analyze Training Overlap — EVGS Patterns by Gait Subtype ===
# These 17 patients have BOTH EVGS scores and gait subtype labels.
# We use them to discover which EVGS items reliably distinguish subtypes.

subtype_stats = defaultdict(lambda: defaultdict(list))
all_train_records = []

for pid in overlap_train:
    t1 = t1_lookup[pid]
    t2 = t2_lookup[pid]
    for side in ['left', 'right']:
        items = t1[side]
        gt = t2[side]['gait_subtype']
        total = items['Total']
        rec = {'pid': pid, 'side': side, 'gt': gt, 'total': total}
        for k in range(1, 18):
            rec[f'i{k}'] = items[str(k)]
            subtype_stats[gt][f'item{k}'].append(items[str(k)])
        subtype_stats[gt]['total'].append(total)
        all_train_records.append(rec)

print('=== EVGS Item Rates by Gait Subtype (from 17 overlap patients, 34 limb-sides) ===\n')
print(f'{"Subtype":>7s} {"N":>3s} {"Total":>6s} | {"i1":>4s} {"i2":>4s} {"i3":>4s} {"i7":>4s} {"i8":>4s} {"i9":>4s} {"i10":>4s} {"i11":>4s} {"i12":>4s} {"i13":>4s}')
print('-' * 85)
for st in ['WNL', 'type1', 'type2', 'type3', 'type4']:
    d = subtype_stats[st]
    n = len(d['total'])
    avg_total = sum(d['total']) / n
    rates = []
    for k in [1, 2, 3, 7, 8, 9, 10, 11, 12, 13]:
        vals = d[f'item{k}']
        rates.append(sum(vals) / n)
    rate_str = ' '.join(f'{r:>4.2f}' for r in rates)
    print(f'{st:>7s} {n:>3d} {avg_total:>6.1f} | {rate_str}')

print('\n=== Key Discriminative Patterns (ABSOLUTE rules from training data) ===')
print('  type1: item11 ALWAYS 0 (knee swing normal), item13 ALWAYS 0 (hip swing normal)')
print('  type2: item1 ALWAYS 1 (no heel contact), item10 ALWAYS 1 (terminal swing abnormal)')
print('  type3: item7 ALWAYS 0 (ankle swing normal), item10 ALWAYS 1')
print('  type4: items 9,10,11,12,13 ALL ALWAYS 1 (everything severely abnormal)')
print('  WNL:   item9=0, item10=0, item11=0, item12=0, item13=0 (proximal all normal)')

print('\n=== L/R Symmetry in Track 2 Training ===')
sym = sum(1 for p in t2_train_data if p['left']['gait_subtype'] == p['right']['gait_subtype'])
print(f'  Symmetric: {sym}/{len(t2_train_data)} ({sym/len(t2_train_data)*100:.0f}%)')
for p in t2_train_data:
    if p['left']['gait_subtype'] != p['right']['gait_subtype']:
        print(f'  Asymmetric: P{p["patient_id"]} L={p["left"]["gait_subtype"]}, R={p["right"]["gait_subtype"]}')

In [ ]:
# === Cell 4: Define Clinical Classification Rules ===
#
# Clinical references (used throughout the rules below):
#
#   [Read 2003]  Read HS, Hazlewood ME, Hillman SJ, Prescott RJ, Robb JE.
#                Edinburgh visual gait score for use in cerebral palsy.
#                J Pediatr Orthop. 2003;23(3):296-301. PMID:12724590.
#                — defines the 17-item EVGS structure and per-item interpretation.
#
#   [R&G 2001]   Rodda JM, Graham HK. Classification of gait patterns in spastic
#                hemiplegia and spastic diplegia: a basis for a management algorithm.
#                Eur J Neurol. 2001;8(Suppl 5):98-108. PMID:11851735.
#                — original Rodda & Graham gait-subtype framework (type1-type4).
#
#   [R&G 2004]   Rodda JM, Graham HK, Carson L, Galea MP, Wolfe R. Sagittal gait
#                patterns in spastic diplegia. J Bone Joint Surg Br. 2004;86(2):251-258.
#                doi:10.1302/0301-620X.86B2.13878.
#                — refined kinematic descriptions: type1 (true equinus) has normal
#                  swing-phase knee/hip; type2 (jump) has stance-phase knee/hip flexion;
#                  type3 (apparent equinus) has normal swing-phase ankle; type4 (crouch)
#                  has severe proximal flexion across stance.
#
#   [Wren 2005]  Wren TAL, Rethlefsen S, Kay RM. Prevalence of specific gait
#                abnormalities in children with cerebral palsy: influence of CP
#                subtype, age, and previous surgery. J Pediatr Orthop. 2005;25(1):79-83.
#                PMID:15614065.
#                — bilateral CP subtype symmetry prior used in Cell 7 L/R post-processing.
#
# These rules use EXCLUSION logic derived from the EVGS reference guide
# [Read 2003] and the Rodda & Graham classification framework
# [R&G 2001 / R&G 2004]. The key insight is that certain EVGS items have
# ABSOLUTE associations with specific subtypes in our training data, and
# those associations are consistent with the kinematic descriptions in
# [R&G 2004]:
#
# - type1 NEVER has item11=1 or item13=1 (swing phase is normal)
#   [R&G 2004]: type1 = true equinus, knee/hip swing within normal range.
# - type3 NEVER has item7=1 (ankle swing is normal)
#   [R&G 2004]: type3 = apparent equinus, ankle DF returns to normal in swing.
# - type4 ALWAYS has items 9,10,11,12,13 = 1 (everything abnormal)
#   [R&G 2004]: type4 = crouch, severe proximal flexion across stance + swing.
# - type2 ALWAYS has item10=1 (terminal swing abnormal)
#   [R&G 2004]: type2 = jump, persistent stance-phase knee + hip flexion;
#   item10 (terminal-swing knee position) is abnormal because the knee fails
#   to extend before initial contact.
# - WNL has proximal items all = 0 and low ankle involvement
#   Derived from [Read 2003] EVGS scoring + WNL exemplars in training cohort.

def classify_clinical(items, verbose=False):
    """
    Classify gait subtype from EVGS binary items using clinical exclusion rules.
    
    The approach uses EXCLUSION first (what the patient CANNOT be), then
    positive evidence (what they are most consistent with).

    Rule sources: [Read 2003] PMID:12724590 (EVGS structure),
    [R&G 2001] PMID:11851735 + [R&G 2004] doi:10.1302/0301-620X.86B2.13878
    (gait pattern definitions). Numeric thresholds (e.g. ankle_score>=2,
    Total<=5) are derived from the 17 cross-track-overlap training patients
    analysed in Cell 3 above; literature provides the EXCLUSION skeleton,
    training data calibrates the thresholds.

    Args:
        items: dict with keys '1'-'17' and 'Total' (EVGS binary items)
        verbose: if True, return detailed reasoning
    
    Returns:
        (subtype, confidence, reasoning_list)
    """
    i1, i2, i3 = items['1'], items['2'], items['3']
    i7 = items['7']
    i8, i9, i10 = items['8'], items['9'], items['10']
    i11, i12, i13 = items['11'], items['12'], items['13']
    total = items['Total']
    
    # Derived features
    ankle_score = i1 + i2 + i3   # 0-3: severity of ankle involvement
    
    reasons = []
    excluded = set()
    
    # === EXCLUSION RULES (from absolute patterns in training data) ===
    
    # If item7=1 (ankle abnormal in swing) -> EXCLUDE type3 (type3 NEVER has i7=1)
    # [R&G 2004]: type3 (apparent equinus) is defined by normal ankle DF in swing.
    if i7 == 1:
        excluded.add('type3')
        reasons.append('i7=1 -> exclude type3 (type3 never has abnormal ankle swing)')
    
    # If item11=1 (knee swing abnormal) -> EXCLUDE type1 (type1 NEVER has i11=1)
    # [R&G 2004]: type1 (true equinus) has near-normal knee swing.
    if i11 == 1:
        excluded.add('type1')
        reasons.append('i11=1 -> exclude type1 (type1 never has abnormal knee swing)')
    
    # If item13=1 (hip swing abnormal) -> EXCLUDE type1 (type1 NEVER has i13=1)
    # [R&G 2004]: type1 has normal hip swing range.
    if i13 == 1:
        excluded.add('type1')
        reasons.append('i13=1 -> exclude type1 (type1 never has abnormal hip swing)')
    
    # If item10=0 (terminal swing normal) -> EXCLUDE type2, type3, type4
    # (all three ALWAYS have i10=1 in training)
    # [R&G 2004]: types 2/3/4 share knee flexion at terminal swing; only type1
    # and WNL routinely show normal item10.
    if i10 == 0:
        excluded.update(['type2', 'type3', 'type4'])
        reasons.append('i10=0 -> exclude type2/3/4 (they always have abnormal terminal swing)')
    
    # === POSITIVE IDENTIFICATION ===
    
    # Check type4 first (most severe, most distinctive pattern)
    # [R&G 2004]: crouch = severe proximal flexion across stance and swing;
    # all 5 proximal items abnormal is the canonical type4 fingerprint.
    if i9 == 1 and i10 == 1 and i11 == 1 and i12 == 1 and i13 == 1:
        if 'type4' not in excluded:
            reasons.append('i9,10,11,12,13 ALL=1 -> type4 (all proximal severely abnormal)')
            return 'type4', 'HIGH', reasons
    
    # Check WNL (very mild involvement)
    # WNL criteria: low total + minimal proximal involvement + low ankle score
    # Training WNL: Total=5, ankle_score=1, i9=0, i10=0, i11=0, i12=0, i13=0
    # Very low Total (<=3) is WNL regardless: insufficient items for any gait pattern
    if total <= 3 and 'WNL' not in excluded:
        reasons.append(f'Total={total}<=3 -> WNL (too mild for any CP gait pattern)')
        return 'WNL', 'HIGH', reasons
    
    if (total <= 5 and i9 == 0 and i10 == 0 and i11 == 0 and i12 == 0 and 
            i13 == 0 and ankle_score <= 1 and 'WNL' not in excluded):
        reasons.append(f'Total={total}<=5, proximal all normal, ankle_score={ankle_score}<=1 -> WNL')
        return 'WNL', 'HIGH', reasons
    
    # Check type1 (True Equinus: swing phase normal, ankle equinus pattern)
    # [R&G 2004]: type1 = ankle equinus + normal swing kinematics + normal hip
    # extension in stance. i11=0 and i13=0 are definitional for true equinus.
    if 'type1' not in excluded and i11 == 0 and i13 == 0:
        # type1: ankle equinus (i1/i2 often abnormal) + normal swing + normal hip stance
        if i10 == 0:
            # i10=0 also excludes type2/3/4 -> must be type1 or WNL
            if total > 5 or ankle_score >= 2:
                reasons.append(f'i11=0,i13=0,i10=0,Total={total},ankle={ankle_score} -> type1')
                return 'type1', 'HIGH', reasons
            else:
                # Low total + low ankle = more like WNL, but WNL check already ran
                reasons.append(f'i11=0,i13=0,i10=0,Total={total}<=5 -> type1 (mild equinus)')
                return 'type1', 'MEDIUM', reasons
        else:
            # i10=1 but swing knee/hip normal. type1 has i10 at 50% in training.
            # Need to differentiate from type2/type3
            if i12 == 0:
                reasons.append('i11=0,i13=0,i12=0 -> type1 (swing+hip stance normal)')
                return 'type1', 'MEDIUM', reasons
            # i12=1 with i11=0,i13=0: ambiguous type1 vs type2
            # type1 mean Total=7.5, type2 mean=10.2
            if total <= 8:
                reasons.append(f'i11=0,i13=0,i12=1,Total={total}<=8 -> type1 (low score)')
                return 'type1', 'LOW', reasons
    
    # Remaining: must be type2 or type3 (or rarely type4)
    # Differentiate type2 vs type3 using [R&G 2004] kinematic distinction:
    # type2 (jump) keeps stance-phase ankle equinus; type3 (apparent equinus)
    # ankle returns toward neutral in swing. i7 is the cleanest discriminator.
    
    if 'type2' not in excluded and 'type3' in excluded:
        reasons.append('type3 excluded -> type2')
        return 'type2', 'MEDIUM', reasons
    
    if 'type3' not in excluded and 'type2' in excluded:
        reasons.append('type2 excluded -> type3')
        return 'type3', 'MEDIUM', reasons
    
    # Neither excluded: use item7 as primary differentiator
    # type3 NEVER has i7=1, type2 has i7=1 in 54% of cases
    if i7 == 1:
        reasons.append('i7=1 -> type2 (ankle abnormal in swing = persistent equinus)')
        return 'type2', 'HIGH', reasons
    
    # Both type2 and type3 possible with i7=0
    # Use item12 (hip extension in stance): type2=0.85, type3=0.56
    if i12 == 1 and i11 == 0:
        reasons.append('i7=0,i12=1,i11=0 -> type2 (hip stance abnormal, knee swing OK)')
        return 'type2', 'MEDIUM', reasons
    
    if i12 == 1 and i11 == 1:
        reasons.append('i7=0,i12=1,i11=1 -> type3 (both knee swing + hip stance abnormal)')
        return 'type3', 'MEDIUM', reasons
    
    if i12 == 0 and i11 == 1:
        reasons.append('i7=0,i12=0,i11=1 -> type3 (knee swing abnormal, no equinus in swing)')
        return 'type3', 'MEDIUM', reasons
    
    # Fallback: i7=0, i12=0, i11=0 but type1 was excluded (so i13=1)
    if i13 == 1:
        reasons.append('i7=0,i12=0,i13=1 -> type2 (hip swing abnormal)')
        return 'type2', 'LOW', reasons
    
    # Should not reach here, but default to type3
    reasons.append('Fallback -> type3')
    return 'type3', 'LOW', reasons

print('Clinical classification function defined.')
print('Uses EXCLUSION rules from absolute patterns in training data + Rodda & Graham framework.')
print('References: Read 2003 PMID:12724590; R&G 2001 PMID:11851735; R&G 2004 doi:10.1302/0301-620X.86B2.13878.')
print()
print('Key reviews from initial version:')
print('  - WNL: Total<=3 is WNL regardless (too mild for any CP gait pattern)')
print('  - WNL: requires ankle_score<=1 (to not misclassify type1 with ankle equinus)')
print('  - type1: ankle_score>=2 can overlay low-total WNL-like patterns')

In [ ]:
# === Cell 5: Validate Clinical Rules on Training Data (17 overlap patients) ===

print('=== Clinical Rule Validation on Training Overlap (34 limb-sides) ===\n')
print(f'{"PID":>4s} {"Side":>5s} | {"GT":>5s} {"Pred":>5s} {"Conf":>6s} | {"Match":>5s} | Reasoning')
print('-' * 110)

correct = 0
total_samples = 0
y_true, y_pred = [], []

for pid in overlap_train:
    t1 = t1_lookup[pid]
    t2 = t2_lookup[pid]
    for side in ['left', 'right']:
        items = t1[side]
        gt = t2[side]['gait_subtype']
        pred, conf, reasons = classify_clinical(items)
        
        match = pred == gt
        correct += int(match)
        total_samples += 1
        y_true.append(gt)
        y_pred.append(pred)
        
        tag = 'OK' if match else 'WRONG'
        reason_str = '; '.join(reasons)
        print(f'P{pid:>3d} {side:>5s} | {gt:>5s} {pred:>5s} {conf:>6s} | {tag:>5s} | {reason_str}')

acc = correct / total_samples
print(f'\nOverall Accuracy: {correct}/{total_samples} = {acc:.3f}')

# Per-class metrics
labels = ['WNL', 'type1', 'type2', 'type3', 'type4']
print(f'\nClassification Report:')
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

print('Confusion Matrix:')
cm = confusion_matrix(y_true, y_pred, labels=labels)
print(f'{"":>8s}', '  '.join(f'{l:>6s}' for l in labels))
for i, row in enumerate(cm):
    print(f'{labels[i]:>8s}', '  '.join(f'{v:>6d}' for v in row))

# Per-patient accuracy (L/R combined)
patient_correct = 0
for pid in overlap_train:
    t2 = t2_lookup[pid]
    t1 = t1_lookup[pid]
    l_pred, _, _ = classify_clinical(t1['left'])
    r_pred, _, _ = classify_clinical(t1['right'])
    l_gt = t2['left']['gait_subtype']
    r_gt = t2['right']['gait_subtype']
    if l_pred == l_gt and r_pred == r_gt:
        patient_correct += 1
print(f'\nPer-patient accuracy (both sides correct): {patient_correct}/{len(overlap_train)} = {patient_correct/len(overlap_train):.3f}')
print(f'\nNote: These rules achieve imperfect accuracy because EVGS binary items')
print(f'lose directional information (e.g., item3=1 can mean either excessive DF')
print(f'or plantarflexion). The rules are used as EVIDENCE, not as the sole classifier.')

In [ ]:
# === Cell 6: Classify Track 2 Test Patients — Detailed Clinical Reasoning ===
#
# Decision pipeline for each test patient:
#   1. Hard exclusions from absolute training-data patterns
#   2. Total score range check against per-subtype training distribution
#   3. Clinical rule classification (exclusion + positive evidence)
#   4. xgb_baseline ML prediction (used in ambiguous low-confidence cases)
#   5. L/R symmetry post-processing

print('=== Detailed Clinical Analysis of Track 2 Test Patients ===\n')

# Total score ranges from training data
total_ranges = {}
for st in ['WNL', 'type1', 'type2', 'type3', 'type4']:
    totals = subtype_stats[st]['total']
    total_ranges[st] = (min(totals), max(totals), sum(totals)/len(totals))
    
print('Total score ranges from training:')
for st, (lo, hi, avg) in sorted(total_ranges.items()):
    print(f'  {st:>5s}: [{lo:>2d}, {hi:>2d}] mean={avg:.1f}')
print()

# Classify each test patient
clinical_predictions = {}

for pid in TRACK2_TEST_IDS:
    print(f'--- Patient {pid} ---')
    
    # Get EVGS items (ground truth or predicted)
    if pid in t1_lookup:
        evgs_source = 'GROUND TRUTH (Track 1 train)'
        items_by_side = {'left': t1_lookup[pid]['left'], 'right': t1_lookup[pid]['right']}
    elif pid in xgb_baseline_t1_preds:
        evgs_source = 'PREDICTED (xgb_baseline Track 1)'
        items_by_side = xgb_baseline_t1_preds[pid]
    else:
        evgs_source = 'NONE'
        items_by_side = None
    
    print(f'  EVGS source: {evgs_source}')
    exp_L = xgb_baseline_t2_preds[pid]['Left']
    exp_R = xgb_baseline_t2_preds[pid]['Right']
    print(f'  xgb_baseline prediction: L={exp_L}, R={exp_R}')
    
    preds = {}
    for side in ['left', 'right']:
        side_label = 'Left' if side == 'left' else 'Right'
        exp_pred = exp_L if side == 'left' else exp_R
        
        if items_by_side is None:
            # No EVGS at all -- keep xgb_baseline
            preds[side] = (exp_pred, 'FALLBACK', ['No EVGS available -> keep xgb_baseline'])
            continue
        
        items = items_by_side[side]
        total = items['Total']
        
        # Apply clinical rules
        rule_pred, rule_conf, rule_reasons = classify_clinical(items)
        
        # Total score sanity check for xgb_baseline prediction
        exp_lo, exp_hi, _ = total_ranges.get(exp_pred, (0, 34, 17))
        total_plausible = (exp_lo - 2) <= total <= (exp_hi + 2)  # allow small margin
        
        # Decision logic
        reasons = list(rule_reasons)
        
        # HIGH-CONFIDENCE OVERLAY: Total score wildly inconsistent with xgb_baseline
        if not total_plausible:
            reasons.append(f'Total={total} OUTSIDE {exp_pred} range [{exp_lo},{exp_hi}] -> overlay xgb_baseline')
            preds[side] = (rule_pred, 'OVERLAY_' + rule_conf, reasons)
        elif rule_conf == 'HIGH' and rule_pred != exp_pred:
            reasons.append(f'HIGH confidence rule disagrees with xgb_baseline ({exp_pred}) -> use rule')
            preds[side] = (rule_pred, rule_conf, reasons)
        elif rule_conf in ['MEDIUM', 'LOW'] and rule_pred != exp_pred:
            # Ambiguous: consider both, lean towards xgb_baseline unless there's exclusion evidence
            i7 = items['7']
            i10, i11, i13 = items['10'], items['11'], items['13']
            exp_excluded = False
            if exp_pred == 'type3' and i7 == 1:
                exp_excluded = True
                reasons.append(f'xgb_baseline={exp_pred} EXCLUDED by i7=1')
            if exp_pred == 'type1' and (i11 == 1 or i13 == 1):
                exp_excluded = True
                reasons.append(f'xgb_baseline={exp_pred} EXCLUDED by i11={i11}/i13={i13}')
            if exp_pred in ['type2', 'type3', 'type4'] and i10 == 0:
                exp_excluded = True
                reasons.append(f'xgb_baseline={exp_pred} EXCLUDED by i10=0')
            if exp_pred == 'WNL' and total > 7:
                exp_excluded = True
                reasons.append(f'xgb_baseline=WNL EXCLUDED by Total={total}>7')
            if exp_pred == 'type2' and total <= 3:
                exp_excluded = True
                reasons.append(f'xgb_baseline=type2 EXCLUDED by Total={total}<=3 (too mild)')
            
            if exp_excluded:
                preds[side] = (rule_pred, 'OVERLAY_' + rule_conf, reasons)
            else:
                reasons.append(f'Rule={rule_pred}({rule_conf}), xgb_baseline={exp_pred} -> keep xgb_baseline (ambiguous)')
                preds[side] = (exp_pred, 'BASELINE', reasons)
        else:
            # Rule agrees with xgb_baseline
            reasons.append(f'Rule agrees with xgb_baseline={exp_pred}')
            preds[side] = (exp_pred, rule_conf, reasons)
        
        items_str = ' '.join(f'i{k}={items[str(k)]}' for k in [1,2,3,7,9,10,11,12,13])
        print(f'  {side_label}: Total={total:>2d} | {items_str}')
        final_pred, final_conf, final_reasons = preds[side]
        print(f'    -> {final_pred} [{final_conf}]')
        for r in final_reasons:
            print(f'       {r}')
    
    clinical_predictions[pid] = {
        'Left': preds['left'],
        'Right': preds['right'],
    }
    print()

In [ ]:
# === Cell 7: L/R Symmetry Post-Processing ===
#
# 91% of training patients have symmetric L/R subtypes.
# When L != R after clinical classification:
#   - If one side has HIGH confidence and the other LOW/MEDIUM, use the HIGH one for both
#   - If both similar confidence, use the one with higher severity (WNL < type1 < type2 < type3 < type4)
#     Rationale: in a CP cohort, under-diagnosis is more likely than over-diagnosis

SEVERITY_ORDER = {'WNL': 0, 'type1': 1, 'type2': 2, 'type3': 3, 'type4': 4}
CONF_ORDER = {'HIGH': 3, 'OVERLAY_HIGH': 3, 'OVERLAY_MEDIUM': 2, 'MEDIUM': 2,
              'OVERLAY_LOW': 1, 'LOW': 1, 'BASELINE': 1, 'FALLBACK': 0}

print('=== L/R Symmetry Post-Processing ===\n')

final_predictions = {}

for pid in TRACK2_TEST_IDS:
    cp = clinical_predictions[pid]
    l_pred, l_conf, l_reasons = cp['Left']
    r_pred, r_conf, r_reasons = cp['Right']
    
    if l_pred == r_pred:
        # Already symmetric
        final_predictions[pid] = {'Left': l_pred, 'Right': r_pred}
        print(f'P{pid:>3d}: L={l_pred:>5s} R={r_pred:>5s} [SYMMETRIC - no change needed]')
    else:
        # Asymmetric - need to resolve
        l_conf_val = CONF_ORDER.get(l_conf, 0)
        r_conf_val = CONF_ORDER.get(r_conf, 0)
        
        if l_conf_val > r_conf_val:
            winner = l_pred
            reason = f'L has higher confidence ({l_conf} > {r_conf})'
        elif r_conf_val > l_conf_val:
            winner = r_pred
            reason = f'R has higher confidence ({r_conf} > {l_conf})'
        else:
            # Same confidence: use higher severity (prefer diagnosing over missing)
            l_sev = SEVERITY_ORDER.get(l_pred, 0)
            r_sev = SEVERITY_ORDER.get(r_pred, 0)
            if l_sev >= r_sev:
                winner = l_pred
                reason = f'Same confidence, L has higher severity ({l_pred} >= {r_pred})'
            else:
                winner = r_pred
                reason = f'Same confidence, R has higher severity ({r_pred} >= {l_pred})'
        
        final_predictions[pid] = {'Left': winner, 'Right': winner}
        print(f'P{pid:>3d}: L={l_pred:>5s}({l_conf}) R={r_pred:>5s}({r_conf}) -> BOTH={winner:>5s} [{reason}]')

print('\n=== Final Track 2 Predictions ===')
for pid in TRACK2_TEST_IDS:
    fp = final_predictions[pid]
    old_l = xgb_baseline_t2_preds[pid]['Left']
    old_r = xgb_baseline_t2_preds[pid]['Right']
    reviews = []
    if fp['Left'] != old_l:
        reviews.append(f'L: {old_l}->{fp["Left"]}')
    if fp['Right'] != old_r:
        reviews.append(f'R: {old_r}->{fp["Right"]}')
    review_str = ' | '.join(reviews) if reviews else 'UNCHANGED'
    print(f'  P{pid:>3d}: L={fp["Left"]:>5s} R={fp["Right"]:>5s}  [{review_str}]')

In [ ]:
# === Cell 7.5: Finalize Clinical Classifier Output ===
# final_predictions is produced by Cell 6 clinical classification plus
# Cell 7 L/R symmetry post-processing.

assert len(final_predictions) == len(TRACK2_TEST_IDS)
print(f'Using clinical classifier outputs for {len(final_predictions)} Track-2 patients')


In [ ]:
# === Cell 8: Generate Submission CSV ===
# Track 1: identical to xgb_baseline (kept for table format)
# Track 2: use clinical rule-based predictions generated above

xgb_baseline_rows = []
with open(xgb_baseline_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        xgb_baseline_rows.append(dict(row))

HEADER = ['ID'] + [f'L{i}' for i in range(1, 18)] + [f'R{i}' for i in range(1, 18)] +          ['Total', 'Left_gait_subtype', 'Right_gait_subtype']

new_rows = []
for row in xgb_baseline_rows:
    if row['ID'].startswith('track1-'):
        new_rows.append(row)

for pid in sorted(TRACK2_TEST_IDS):
    fp = final_predictions[pid]
    row = {'ID': f'track2-{pid}'}
    for i in range(1, 18):
        row[f'L{i}'] = -1
        row[f'R{i}'] = -1
    row['Total'] = -1
    row['Left_gait_subtype'] = fp['Left']
    row['Right_gait_subtype'] = fp['Right']
    new_rows.append(row)

df_sub = pd.DataFrame(new_rows, columns=HEADER)

sub_path = os.path.join(SUBMISSIONS_DIR, 'track2_clinical_predictions.csv')
df_sub.to_csv(sub_path, index=False)

print(f'Saved Track-2 clinical predictions: {sub_path}')
print(f'Shape: {df_sub.shape}')

assert list(df_sub.columns) == HEADER, 'Column mismatch!'
t1_rows = df_sub[df_sub['ID'].str.startswith('track1-')]
t2_rows = df_sub[df_sub['ID'].str.startswith('track2-')]
assert len(t1_rows) == 16, f'Expected 16 Track 1 rows, got {len(t1_rows)}'
assert len(t2_rows) == 9, f'Expected 9 Track 2 rows, got {len(t2_rows)}'
print(f'Verification: Track1={len(t1_rows)} rows, Track2={len(t2_rows)} rows - OK')


In [ ]:
# === Cell 9: Comparison - track2_clinical vs xgb_baseline ===

print('=' * 70)
print('  track2_clinical vs xgb_baseline: Track 2 Prediction Comparison')
print('=' * 70)
print()
print(f'{"PID":>4s} | {"xgb_L":>8s} {"xgb_R":>8s} | {"clinical_L":>10s} {"clinical_R":>10s} | Changes')
print('-' * 78)

n_reviews = 0
n_limb_reviews = 0
for pid in TRACK2_TEST_IDS:
    old_l = xgb_baseline_t2_preds[pid]['Left']
    old_r = xgb_baseline_t2_preds[pid]['Right']
    new_l = final_predictions[pid]['Left']
    new_r = final_predictions[pid]['Right']
    reviews = []
    if old_l != new_l:
        reviews.append(f'L:{old_l}->{new_l}')
        n_limb_reviews += 1
    if old_r != new_r:
        reviews.append(f'R:{old_r}->{new_r}')
        n_limb_reviews += 1
    if reviews:
        n_reviews += 1
    review_str = ', '.join(reviews) if reviews else '-'
    print(f'P{pid:>3d} | {old_l:>8s} {old_r:>8s} | {new_l:>10s} {new_r:>10s} | {review_str}')

print()
print(f'Total patients changed from xgb_baseline: {n_reviews}/9')
print(f'Limb-side change count: {n_limb_reviews}/18')
print('Clinical classifier inputs: released Track-1 train EVGS when available; otherwise Track-1 model predictions.')
print('Track-2 labels are generated by the clinical classifier in this notebook.')


In [ ]:
# === Cell 10: Summary ===

print('=' * 70)
print('  Track-2 Clinical Classifier -- Summary')
print('=' * 70)
print()
print('  Approach:')
print('    - Track 1 rows are kept only for submission-table format')
print('    - Track 2 subtypes are produced by clinical rules applied to EVGS scores')
print('    - Released Track-1 train EVGS is used only for cross-track patients with labels')
print('    - Remaining Track-2 patients use Track-1 model-predicted EVGS')
print('    - Track-2 test subtype labels are generated by this notebook')
print()
print('  Clinical Rule Framework:')
print('    - Exclusion logic from Rodda & Graham / training patterns')
print('    - Total score range consistency checks')
print('    - L/R symmetry post-processing from training-cohort symmetry')
print()
print(f'  Submission: {sub_path}')
print(f'  Rows: {len(df_sub)} (Track1: 16, Track2: 9)')
print('=' * 70)
